# Emoji Toxicity Detection — Data Analysis & Results

This notebook explores the knowledge base and visualizes evaluation results.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

DATA_DIR = Path("..") / "data"
KB_PATH = DATA_DIR / "processed" / "knowledge_base.jsonl"
RESULTS_PATH = DATA_DIR / "results" / "eval_results.json"

## 1. Knowledge Base Analysis

In [ ]:
# Load KB
entries = []
if KB_PATH.exists():
    with open(KB_PATH) as f:
        entries = [json.loads(line) for line in f if line.strip()]
    print(f"Knowledge base: {len(entries)} entries")
else:
    print("KB not built yet. Run: python -m scripts.build_knowledge_base")

if entries:
    df = pd.DataFrame(entries)
    print(f"\nRisk category distribution:")
    print(df['risk_category'].value_counts())
    
    print(f"\nSources:")
    all_sources = [s for sources in df['sources'] for s in sources]
    print(Counter(all_sources).most_common())

In [ ]:
# Visualize risk categories
if entries:
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    df['risk_category'].value_counts().plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('Knowledge Base — Risk Category Distribution')
    ax.set_ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 2. Evaluation Results

In [ ]:

# Load evaluation results (new schema: config + results with bootstrap CIs, optional cross-seed aggregation)
if RESULTS_PATH.exists():
    with open(RESULTS_PATH) as f:
        payload = json.load(f)

    config = payload.get("config", {})
    print(f"Config: {config}\n")

    rows = []
    for r in payload["results"]:
        seed0 = r["seed_0"]
        row = {
            "method": r["name"],
            "n_samples": seed0["n_samples"],
        }
        for metric in ["accuracy", "precision", "recall", "f1_macro", "auroc"]:
            m = seed0.get(metric)
            if m is None:
                row[metric] = None
                row[f"{metric}_ci"] = None
            else:
                row[metric] = m["value"]
                row[f"{metric}_ci"] = (
                    (m.get("ci_low"), m.get("ci_high"))
                    if m.get("ci_low") is not None
                    else None
                )
        if "across_seeds" in r:
            for metric in ["accuracy", "f1_macro", "auroc"]:
                agg = r["across_seeds"].get(metric)
                if agg:
                    row[f"{metric}_mean_std"] = f"{agg['mean']:.3f} ± {agg['std']:.3f}"
        rows.append(row)

    results_df = pd.DataFrame(rows)
    print(results_df.to_string(index=False))

    # Bar chart with CI error bars
    fig, ax = plt.subplots(figsize=(10, 6))
    metrics_to_plot = ["accuracy", "precision", "recall", "f1_macro"]
    n_methods = len(results_df)
    x = range(n_methods)
    width = 0.2

    for i, metric in enumerate(metrics_to_plot):
        vals = results_df[metric].tolist()
        cis = results_df[f"{metric}_ci"].tolist()
        yerr_low = [v - c[0] if c else 0 for v, c in zip(vals, cis)]
        yerr_high = [c[1] - v if c else 0 for v, c in zip(vals, cis)]
        ax.bar(
            [xi + i * width for xi in x],
            vals,
            width,
            yerr=[yerr_low, yerr_high],
            label=metric,
            capsize=3,
        )

    ax.set_xticks([xi + 1.5 * width for xi in x])
    ax.set_xticklabels(results_df["method"])
    ax.set_ylim(0, 1)
    ax.legend()
    ax.set_title("Evaluation: Method Comparison (bootstrap 95% CIs)")
    plt.tight_layout()
    plt.show()
else:
    print("No results yet. Run: python -m scripts.evaluate")
